<a href="https://colab.research.google.com/github/N-S-Wickramanayaka/Brahmee-transformer/blob/main/Brahmi_character_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Install all required packages
!pip install timm==0.9.12          # For Visformer and model utilities
!pip install albumentations==1.3.1 # For advanced image augmentations
!pip install scikit-learn==1.3.2   # For metrics and splitting
!pip install matplotlib==3.7.5     # For plotting
!pip install seaborn==0.13.1       # For confusion matrix
!pip install tqdm                  # For progress bars
!pip install opencv-python-headless # For image processing

print("All packages installed successfully!")

In [ ]:
!pip install kaggle


In [ ]:
# Cell 3: Setup Kaggle API
# First, go to kaggle.com → Your Profile → Account → Create New API Token
# This downloads a kaggle.json file

from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

In [ ]:
# Cell 4: Configure Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API configured!")

In [ ]:
# Cell 5: Download and extract dataset
!kaggle datasets download -d vajirj/sinhala-early-brahmi-inscription-dataset
!unzip -q sinhala-early-brahmi-inscription-dataset.zip -d ./brahmi_dataset
print("Dataset downloaded and extracted!")

In [ ]:
# Cell 6: Explore dataset structure
import os

dataset_path = './brahmi_dataset'

# Find the actual root directory containing class folders
def find_class_root(path):
    """Recursively find the directory that contains class subdirectories with images"""
    for root, dirs, files_list in os.walk(path):
        # Check if subdirectories contain image files
        has_image_subdirs = False
        for d in dirs:
            subdir_path = os.path.join(root, d)
            sub_files = os.listdir(subdir_path)
            image_files = [f for f in sub_files if f.lower().endswith(
                ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp'))]
            if len(image_files) > 0:
                has_image_subdirs = True
                break
        if has_image_subdirs and len(dirs) > 10:  # Should have many class folders
            return root
    return path

DATA_ROOT = find_class_root(dataset_path)
print(f"Data root directory: {DATA_ROOT}")

# List all classes
classes = sorted([d for d in os.listdir(DATA_ROOT)
                  if os.path.isdir(os.path.join(DATA_ROOT, d))])
print(f"\nTotal number of classes: {len(classes)}")
print(f"\nFirst 20 classes: {classes[:20]}")

# Count images per class
class_counts = {}
total_images = 0
for cls in classes:
    cls_path = os.path.join(DATA_ROOT, cls)
    count = len([f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp'))])
    class_counts[cls] = count
    total_images += count

print(f"\nTotal images: {total_images}")
print(f"Average images per class: {total_images / len(classes):.1f}")
print(f"Min images in a class: {min(class_counts.values())}")
print(f"Max images in a class: {max(class_counts.values())}")

In [ ]:
# Cell 7: Visualize sample images from different classes
import matplotlib.pyplot as plt
from PIL import Image
import random

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle('Sample Images from Different Classes', fontsize=16)

sample_classes = random.sample(classes, min(32, len(classes)))

for idx, (ax, cls) in enumerate(zip(axes.flatten(), sample_classes)):
    cls_path = os.path.join(DATA_ROOT, cls)
    images = [f for f in os.listdir(cls_path)
              if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp'))]
    if images:
        img_path = os.path.join(cls_path, random.choice(images))
        try:
            img = Image.open(img_path).convert('RGB')
            ax.imshow(img)
            ax.set_title(cls[:15], fontsize=8)
        except:
            ax.text(0.5, 0.5, 'Error', ha='center')
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sample visualization saved!")

In [ ]:
# Cell 8: Plot class distribution
fig, ax = plt.subplots(figsize=(20, 6))
sorted_counts = dict(sorted(class_counts.items(), key=lambda x: x[1], reverse=True))
bars = ax.bar(range(len(sorted_counts)), list(sorted_counts.values()), color='steelblue')
ax.set_xlabel('Class Index')
ax.set_ylabel('Number of Images')
ax.set_title('Distribution of Images Across Classes')
ax.axhline(y=total_images/len(classes), color='red', linestyle='--', label='Average')
ax.legend()
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify imbalanced classes
print("\nClasses with fewer than 10 images:")
for cls, count in sorted_counts.items():
    if count < 10:
        print(f"  {cls}: {count} images")

In [ ]:
# Cell 9: Import all necessary libraries
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("All libraries imported and seed set!")

In [ ]:
# Cell 10: Build the full dataset file list
image_paths = []
image_labels = []

for cls in classes:
    cls_path = os.path.join(DATA_ROOT, cls)
    for img_name in os.listdir(cls_path):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')):
            image_paths.append(os.path.join(cls_path, img_name))
            image_labels.append(cls)

# Encode labels to integers
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(image_labels)
num_classes = len(label_encoder.classes_)

print(f"Total images found: {len(image_paths)}")
print(f"Number of classes: {num_classes}")
print(f"Label encoding: {label_encoder.classes_[:10]}... → [0, 1, 2, ...]")

In [ ]:
# Cell 11: Split dataset - 70% train, 15% validation, 15% test
# Use stratified splitting to maintain class distribution

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths, encoded_labels,
    test_size=0.30,
    stratify=encoded_labels,
    random_state=42
)

# Second split: 50% of temp = 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f"Training set:   {len(X_train)} images")
print(f"Validation set: {len(X_val)} images")
print(f"Test set:       {len(X_test)} images")
print(f"Total:          {len(X_train) + len(X_val) + len(X_test)} images")

In [ ]:
# Cell 12: Define image transforms
# Visformer typically uses 224x224 input
IMG_SIZE = 224

# Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),  # Resize slightly larger
    transforms.RandomCrop(IMG_SIZE),                     # Random crop to target size
    transforms.RandomHorizontalFlip(p=0.3),             # Light horizontal flip
    transforms.RandomRotation(degrees=15),               # Slight rotation
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),                            # Slight translation
        scale=(0.9, 1.1),                                # Slight scaling
    ),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2,
    ),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet normalization
        std=[0.229, 0.224, 0.225]
    ),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),  # Random erasing
])

# Validation/Test transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print("Transforms defined!")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")

In [ ]:
# Cell 13: Custom Dataset class
class BrahmiDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # Return a blank image if loading fails
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# Create datasets
train_dataset = BrahmiDataset(X_train, y_train, transform=train_transform)
val_dataset = BrahmiDataset(X_val, y_val, transform=val_transform)
test_dataset = BrahmiDataset(X_test, y_test, transform=val_transform)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")
print(f"Test dataset:  {len(test_dataset)} samples")

In [ ]:
# Cell 14: Create weighted sampler for imbalanced classes
from collections import Counter

# Count samples per class in training set
train_class_counts = Counter(y_train)
total_train = len(y_train)

# Calculate weight for each class (inverse frequency)
class_weights = {cls: total_train / count for cls, count in train_class_counts.items()}

# Assign weight to each sample
sample_weights = [class_weights[label] for label in y_train]
sample_weights = torch.tensor(sample_weights, dtype=torch.float)

# Create sampler
weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("Weighted sampler created to handle class imbalance!")
print(f"Min class weight: {min(class_weights.values()):.2f}")
print(f"Max class weight: {max(class_weights.values()):.2f}")

In [ ]:
# Cell 15 (FIXED): Smaller batch size
BATCH_SIZE = 16  # Reduced from 32 to 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=weighted_sampler,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Batch size reduced to: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
# Cell 16: Visualize augmented images
def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """Reverse the normalization for visualization"""
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return torch.clamp(tensor, 0, 1)

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle('Augmented Training Images', fontsize=14)

for idx in range(16):
    ax = axes[idx // 8][idx % 8]
    img = denormalize(batch_images[idx])
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f'Class: {batch_labels[idx].item()}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('augmented_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 17: Visformer Architecture Explanation
print("""
╔══════════════════════════════════════════════════════════════╗
║                  VISFORMER ARCHITECTURE                       ║
╠══════════════════════════════════════════════════════════════╣
║                                                               ║
║  Input Image (224×224×3)                                      ║
║       │                                                       ║
║       ▼                                                       ║
║  ┌─────────────────┐                                          ║
║  │  Stem (Conv)     │  ← Initial feature extraction           ║
║  │  Patch Embedding │  ← Convert to patch tokens              ║
║  └────────┬────────┘                                          ║
║           ▼                                                   ║
║  ┌─────────────────┐                                          ║
║  │  Stage 1        │  ← Conv blocks (spatial processing)      ║
║  │  (Conv Blocks)  │  ← Keeps spatial structure               ║
║  └────────┬────────┘                                          ║
║           ▼                                                   ║
║  ┌─────────────────┐                                          ║
║  │  Stage 2        │  ← Conv blocks (deeper features)         ║
║  │  (Conv Blocks)  │                                          ║
║  └────────┬────────┘                                          ║
║           ▼                                                   ║
║  ┌─────────────────┐                                          ║
║  │  Stage 3        │  ← Transformer blocks (global attention) ║
║  │  (Transformer)  │  ← Self-attention mechanism              ║
║  └────────┬────────┘                                          ║
║           ▼                                                   ║
║  ┌─────────────────┐                                          ║
║  │  Global Avg Pool│  ← Aggregate spatial features            ║
║  │  + FC Head      │  ← Classification head                   ║
║  └────────┬────────┘                                          ║
║           ▼                                                   ║
║    Output: 110 classes                                        ║
║                                                               ║
╚══════════════════════════════════════════════════════════════╝
""")

In [ ]:
# Cell 18: Visformer components - Building blocks

import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial


class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample."""
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.:
            return x
        keep_prob = 1 - self.drop_prob
        # Create random tensor
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor = torch.floor(random_tensor + keep_prob)
        output = x / keep_prob * random_tensor
        return output


class BatchNorm2d(nn.BatchNorm2d):
    """BatchNorm2d that can handle both 4D (BCHW) and 3D (BNC) tensors."""
    def forward(self, x):
        if x.dim() == 4:
            return super().forward(x)
        # Handle 3D tensor: (B, N, C) -> (B, C, N) -> BN -> (B, N, C)
        return super().forward(x.transpose(1, 2)).transpose(1, 2)


class Mlp(nn.Module):
    """MLP block using 1x1 convolutions (spatial format) or linear layers."""
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0., group=8, spatial_conv=False):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.spatial_conv = spatial_conv

        if spatial_conv:
            # Use 1x1 conv for spatial format (B, C, H, W)
            self.hidden_features = hidden_features
            self.fc1 = nn.Conv2d(in_features, hidden_features, 1, stride=1, padding=0, bias=False)
            self.bn1 = nn.BatchNorm2d(hidden_features)
            # Depthwise conv for spatial mixing
            self.dw_conv = nn.Conv2d(hidden_features, hidden_features, 3, stride=1, padding=1,
                                      groups=hidden_features, bias=False)
            self.bn_dw = nn.BatchNorm2d(hidden_features)
            self.fc2 = nn.Conv2d(hidden_features, out_features, 1, stride=1, padding=0, bias=False)
            self.bn2 = nn.BatchNorm2d(out_features)
        else:
            self.fc1 = nn.Linear(in_features, hidden_features)
            self.fc2 = nn.Linear(hidden_features, out_features)

        self.act = act_layer()
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        if self.spatial_conv:
            x = self.bn1(self.fc1(x))
            x = self.act(x)
            x = self.drop(x)
            x = self.bn_dw(self.dw_conv(x))
            x = self.act(x)
            x = self.drop(x)
            x = self.bn2(self.fc2(x))
            x = self.drop(x)
        else:
            x = self.fc1(x)
            x = self.act(x)
            x = self.drop(x)
            x = self.fc2(x)
            x = self.drop(x)
        return x


class Attention(nn.Module):
    """Multi-head self-attention for spatial (BCHW) or sequence (BNC) format."""
    def __init__(self, dim, num_heads=8, head_dim=None, qkv_bias=False,
                 attn_drop=0., proj_drop=0., spatial_conv=False):
        super().__init__()
        self.spatial_conv = spatial_conv
        self.num_heads = num_heads
        self.head_dim = head_dim or dim // num_heads
        self.scale = self.head_dim ** -0.5
        inner_dim = self.head_dim * num_heads

        if spatial_conv:
            self.qkv = nn.Conv2d(dim, inner_dim * 3, 1, bias=qkv_bias)
            self.proj = nn.Conv2d(inner_dim, dim, 1)
        else:
            self.qkv = nn.Linear(dim, inner_dim * 3, bias=qkv_bias)
            self.proj = nn.Linear(inner_dim, dim)

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        if self.spatial_conv:
            B, C, H, W = x.shape
            N = H * W
            qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, N)
            qkv = qkv.permute(1, 0, 2, 4, 3)  # 3, B, heads, N, head_dim
            q, k, v = qkv.unbind(0)
        else:
            B, N, C = x.shape
            qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
            qkv = qkv.permute(2, 0, 3, 1, 4)  # 3, B, heads, N, head_dim
            q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        out = (attn @ v)  # B, heads, N, head_dim

        if self.spatial_conv:
            out = out.permute(0, 1, 3, 2).reshape(B, -1, H, W)
            out = self.proj(out)
        else:
            out = out.transpose(1, 2).reshape(B, N, -1)
            out = self.proj(out)

        out = self.proj_drop(out)
        return out


class SpatialBlock(nn.Module):
    """Block that operates in spatial (BCHW) format - used in early stages."""
    def __init__(self, dim, num_heads, head_dim=None, mlp_ratio=4.,
                 qkv_bias=False, drop=0., attn_drop=0., drop_path=0.,
                 act_layer=nn.GELU, group=8, attn_disabled=False):
        super().__init__()
        self.attn_disabled = attn_disabled
        mlp_hidden = int(dim * mlp_ratio)

        if not attn_disabled:
            self.norm1 = nn.BatchNorm2d(dim)
            self.attn = Attention(dim, num_heads=num_heads, head_dim=head_dim,
                                  qkv_bias=qkv_bias, attn_drop=attn_drop,
                                  proj_drop=drop, spatial_conv=True)
            self.drop_path1 = DropPath(drop_path) if drop_path > 0. else nn.Identity()

        self.norm2 = nn.BatchNorm2d(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden,
                       act_layer=act_layer, drop=drop, group=group,
                       spatial_conv=True)
        self.drop_path2 = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        if not self.attn_disabled:
            x = x + self.drop_path1(self.attn(self.norm1(x)))
        x = x + self.drop_path2(self.mlp(self.norm2(x)))
        return x


class TransformerBlock(nn.Module):
    """Standard transformer block operating on sequence (B, N, C) format."""
    def __init__(self, dim, num_heads, head_dim=None, mlp_ratio=4.,
                 qkv_bias=False, drop=0., attn_drop=0., drop_path=0.,
                 act_layer=nn.GELU):
        super().__init__()
        mlp_hidden = int(dim * mlp_ratio)

        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads=num_heads, head_dim=head_dim,
                              qkv_bias=qkv_bias, attn_drop=attn_drop,
                              proj_drop=drop, spatial_conv=False)
        self.drop_path1 = DropPath(drop_path) if drop_path > 0. else nn.Identity()

        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden,
                       act_layer=act_layer, drop=drop, spatial_conv=False)
        self.drop_path2 = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        x = x + self.drop_path1(self.attn(self.norm1(x)))
        x = x + self.drop_path2(self.mlp(self.norm2(x)))
        return x


print("All building blocks defined!")

In [ ]:
# Cell 19: Complete Visformer Model

class Visformer(nn.Module):
    """
    Visformer: The Vision-friendly Transformer

    Architecture:
    - Stem: Convolutional stem for initial feature extraction
    - Stage 1: Spatial (Conv) blocks - processes features in BCHW format
    - Stage 2: Spatial (Conv) blocks - deeper spatial processing
    - Stage 3: Transformer blocks - global self-attention in BNC format
    - Head: Global average pooling + classification layer
    """
    def __init__(
        self,
        img_size=224,
        in_channels=3,
        num_classes=110,
        embed_dim=192,       # Base embedding dimension
        depth=(3, 4, 8),     # Number of blocks in each stage
        num_heads=(3, 6, 12),# Attention heads per stage
        mlp_ratio=4.0,
        qkv_bias=True,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.1,  # Stochastic depth
        group=8,
        attn_stage='1-1-1',  # Which stages use attention
    ):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim

        # Parse attention stage config
        # '1-1-1' means all stages use attention
        # '0-0-1' means only stage 3 uses attention
        attn_flags = [int(x) for x in attn_stage.split('-')]

        # Stage dimensions: each stage doubles the channels
        stage1_dim = embed_dim       # 192
        stage2_dim = embed_dim * 2   # 384
        stage3_dim = embed_dim * 4   # 768

        # ==================== STEM ====================
        # Convolutional stem to extract initial features and create patches
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, stage1_dim, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stage1_dim),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),  # 224 -> 56
        )

        # ==================== STAGE 1 ====================
        # Spatial blocks processing at 56x56 resolution
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depth))]
        dpr_idx = 0

        stage1_blocks = []
        for i in range(depth[0]):
            stage1_blocks.append(
                SpatialBlock(
                    dim=stage1_dim,
                    num_heads=num_heads[0],
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    drop=drop_rate,
                    attn_drop=attn_drop_rate,
                    drop_path=dpr[dpr_idx],
                    group=group,
                    attn_disabled=(attn_flags[0] == 0),
                )
            )
            dpr_idx += 1
        self.stage1 = nn.Sequential(*stage1_blocks)

        # ==================== TRANSITION 1→2 ====================
        # Downsample: 56x56 -> 28x28, channels: stage1_dim -> stage2_dim
        self.transition1 = nn.Sequential(
            nn.Conv2d(stage1_dim, stage2_dim, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(stage2_dim),
            nn.ReLU(inplace=True),
        )

        # ==================== STAGE 2 ====================
        # Spatial blocks processing at 28x28 resolution
        stage2_blocks = []
        for i in range(depth[1]):
            stage2_blocks.append(
                SpatialBlock(
                    dim=stage2_dim,
                    num_heads=num_heads[1],
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    drop=drop_rate,
                    attn_drop=attn_drop_rate,
                    drop_path=dpr[dpr_idx],
                    group=group,
                    attn_disabled=(attn_flags[1] == 0),
                )
            )
            dpr_idx += 1
        self.stage2 = nn.Sequential(*stage2_blocks)

        # ==================== TRANSITION 2→3 ====================
        # Downsample: 28x28 -> 14x14, channels: stage2_dim -> stage3_dim
        self.transition2 = nn.Sequential(
            nn.Conv2d(stage2_dim, stage3_dim, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(stage3_dim),
            nn.ReLU(inplace=True),
        )

        # ==================== STAGE 3 ====================
        # Transformer blocks - flatten spatial dims to sequence
        # At 14x14 resolution, sequence length = 196 tokens
        self.pos_embed = nn.Parameter(torch.zeros(1, 196, stage3_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.pos_drop = nn.Dropout(p=drop_rate)

        stage3_blocks = []
        for i in range(depth[2]):
            stage3_blocks.append(
                TransformerBlock(
                    dim=stage3_dim,
                    num_heads=num_heads[2],
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    drop=drop_rate,
                    attn_drop=attn_drop_rate,
                    drop_path=dpr[dpr_idx],
                )
            )
            dpr_idx += 1
        self.stage3 = nn.Sequential(*stage3_blocks)

        # ==================== CLASSIFICATION HEAD ====================
        self.norm = nn.LayerNorm(stage3_dim)
        self.head = nn.Sequential(
            nn.Linear(stage3_dim, stage3_dim // 2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(stage3_dim // 2, num_classes),
        )

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, (nn.LayerNorm, nn.BatchNorm2d)):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward_features(self, x):
        # Stem: (B, 3, 224, 224) -> (B, 192, 56, 56)
        x = self.stem(x)

        # Stage 1: (B, 192, 56, 56) -> (B, 192, 56, 56)
        x = self.stage1(x)

        # Transition 1: (B, 192, 56, 56) -> (B, 384, 28, 28)
        x = self.transition1(x)

        # Stage 2: (B, 384, 28, 28) -> (B, 384, 28, 28)
        x = self.stage2(x)

        # Transition 2: (B, 384, 28, 28) -> (B, 768, 14, 14)
        x = self.transition2(x)

        # Reshape for transformer: (B, 768, 14, 14) -> (B, 196, 768)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, 196, 768)

        # Add positional embedding
        x = x + self.pos_embed
        x = self.pos_drop(x)

        # Stage 3 (Transformer): (B, 196, 768) -> (B, 196, 768)
        x = self.stage3(x)

        # Normalize
        x = self.norm(x)

        # Global average pooling: (B, 196, 768) -> (B, 768)
        x = x.mean(dim=1)

        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        return x


print("Visformer model class defined!")

In [ ]:
# Cell 20 (BEST OPTION): Pretrained Visformer-Small using timm
import gc
import torch
import timm

# Clear any existing model
try:
    del model
except:
    pass
gc.collect()
torch.cuda.empty_cache()

# Create PRETRAINED Visformer-Small
model = timm.create_model(
    'visformer_small',           # Visformer-Small architecture
    pretrained=True,             # Load ImageNet pretrained weights
    num_classes=num_classes,     # Your 110 classes
    drop_rate=0.1,               # Dropout for regularization
    drop_path_rate=0.1,          # Stochastic depth
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Verify
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Loaded: Pretrained Visformer-Small")
print(f"Device: {device}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass
dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    output = model(dummy)
print(f"\nInput shape:  {dummy.shape}")
print(f"Output shape: {output.shape}")
print(f"GPU Memory: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

In [ ]:
# Cell 21: Print detailed model structure
print("=" * 70)
print("VISFORMER-SMALL ARCHITECTURE SUMMARY")
print("=" * 70)

print("\n📌 STEM (Convolutional Feature Extraction):")
print(f"   Input:  (B, 3, 224, 224)")
print(f"   Output: (B, 192, 56, 56)")
for name, module in model.stem.named_children():
    if hasattr(module, 'weight'):
        print(f"   [{name}] {module.__class__.__name__}: {list(module.weight.shape)}")

print(f"\n📌 STAGE 1 (Spatial Blocks × 3):")
print(f"   Resolution: 56×56, Channels: 192")
params_s1 = sum(p.numel() for p in model.stage1.parameters())
print(f"   Parameters: {params_s1:,}")

print(f"\n📌 TRANSITION 1→2:")
print(f"   56×56 → 28×28, 192 → 384 channels")

print(f"\n📌 STAGE 2 (Spatial Blocks × 4):")
print(f"   Resolution: 28×28, Channels: 384")
params_s2 = sum(p.numel() for p in model.stage2.parameters())
print(f"   Parameters: {params_s2:,}")

print(f"\n📌 TRANSITION 2→3:")
print(f"   28×28 → 14×14, 384 → 768 channels")

print(f"\n📌 STAGE 3 (Transformer Blocks × 8):")
print(f"   Sequence: 196 tokens, Dimension: 768")
print(f"   Attention Heads: 12")
params_s3 = sum(p.numel() for p in model.stage3.parameters())
print(f"   Parameters: {params_s3:,}")

print(f"\n📌 CLASSIFICATION HEAD:")
print(f"   768 → 384 → {num_classes}")
params_head = sum(p.numel() for p in model.head.parameters())
print(f"   Parameters: {params_head:,}")
print("=" * 70)

In [ ]:
# Cell 22: Loss function with label smoothing
class LabelSmoothingCrossEntropy(nn.Module):
    """
    Cross entropy with label smoothing.
    Helps prevent overconfident predictions and improves generalization.
    """
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, pred, target):
        logprobs = F.log_softmax(pred, dim=-1)
        nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth_loss = -logprobs.mean(dim=-1)
        loss = self.confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()

# Create loss function
criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
print("Loss function: Label Smoothing Cross Entropy (smoothing=0.1)")

In [ ]:
# Cell 23 (UPDATED): Use lower learning rate for pretrained model
optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-5,            # Lower LR (was 1e-4) - pretrained needs gentler updates
    weight_decay=0.05,
    betas=(0.9, 0.999),
)

NUM_EPOCHS = 50          # Fewer epochs needed (was 100) - pretrained converges fast
WARMUP_EPOCHS = 3

scheduler = WarmupCosineScheduler(optimizer, WARMUP_EPOCHS, NUM_EPOCHS)
print(f"Optimizer: AdamW (lr=3e-5)")
print(f"Training for {NUM_EPOCHS} epochs (pretrained converges faster)")

In [ ]:
# Cell 24: Training and validation functions

def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [Train]', leave=False)
    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def validate(model, loader, criterion, device, epoch, phase='Val'):
    """Validate/Test the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [{phase}]', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels


print("Training and validation functions defined!")

In [ ]:
# Cell 25: Early stopping class
class EarlyStopping:
    """Stop training when validation loss doesn't improve."""
    def __init__(self, patience=15, min_delta=0.001, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        if self.mode == 'min':
            improved = score < (self.best_score - self.min_delta)
        else:
            improved = score > (self.best_score + self.min_delta)

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        return False

early_stopping = EarlyStopping(patience=15, mode='min')
print("Early stopping configured with patience=15")

In [ ]:
# Cell 26: MAIN TRAINING LOOP
import time

# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'lr': []
}

best_val_acc = 0.0
best_val_loss = float('inf')
best_epoch = 0

print("=" * 70)
print(f"Starting Training: {NUM_EPOCHS} epochs")
print(f"Model: Visformer-Small | Classes: {num_classes}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print("=" * 70)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    # Update learning rate
    current_lr = scheduler.step(epoch)
    history['lr'].append(current_lr)

    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device, epoch
    )

    # Validate
    val_loss, val_acc, val_preds, val_labels = validate(
        model, val_loader, criterion, device, epoch
    )

    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    epoch_time = time.time() - epoch_start

    # Print epoch results
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
          f"LR: {current_lr:.6f} | Time: {epoch_time:.1f}s")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = epoch + 1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'num_classes': num_classes,
            'label_encoder_classes': label_encoder.classes_.tolist(),
        }, 'best_model.pth')
        print(f"  ★ New best model saved! Val Acc: {val_acc:.2f}%")

    # Early stopping check
    if early_stopping(val_loss):
        print(f"\n⚠ Early stopping triggered at epoch {epoch+1}")
        break

total_time = time.time() - start_time
print("\n" + "=" * 70)
print(f"Training Complete!")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Best Epoch: {best_epoch} | Best Val Acc: {best_val_acc:.2f}%")
print("=" * 70)

In [ ]:
# Cell A: Clear GPU memory
import torch
import gc

# Delete model and optimizer if they exist
try:
    del model
    del optimizer
    del scheduler
except:
    pass

# Clear cache
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Check available memory
free_mem = torch.cuda.mem_get_info(0)[0] / 1e9
total_mem = torch.cuda.mem_get_info(0)[1] / 1e9
print(f"Free GPU Memory: {free_mem:.2f} GB / {total_mem:.2f} GB")

In [ ]:
# Cell 27: Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss', color='blue', alpha=0.8)
axes[0].plot(history['val_loss'], label='Val Loss', color='red', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history['train_acc'], label='Train Acc', color='blue', alpha=0.8)
axes[1].plot(history['val_acc'], label='Val Acc', color='red', alpha=0.8)
axes[1].axhline(y=best_val_acc, color='green', linestyle='--', label=f'Best: {best_val_acc:.2f}%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(history['lr'], color='purple', alpha=0.8)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 28 (FIXED): Load best model and evaluate on test set
import torch
import gc
from tqdm import tqdm
from torch.amp import autocast

gc.collect()
torch.cuda.empty_cache()

# Load best model - ADD weights_only=False
checkpoint = torch.load('best_model.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"   Best validation accuracy: {checkpoint['val_acc']:.2f}%")

# Evaluate on test set with mixed precision
model.eval()
test_loss = 0.0
correct = 0
total = 0
test_preds = []
test_labels = []

with torch.no_grad():
    pbar = tqdm(test_loader, desc='Testing', leave=True)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        test_preds.extend(predicted.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })

test_loss = test_loss / total
test_acc = 100. * correct / total

print(f"\n{'='*50}")
print(f"TEST SET RESULTS")
print(f"{'='*50}")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Total Test Samples: {total}")
print(f"Correct Predictions: {correct}")
print(f"{'='*50}")

In [ ]:
# Cell 29: Classification report
# Get class names
class_names = label_encoder.classes_

# Full classification report
report = classification_report(
    test_labels, test_preds,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

# Print summary
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT (Summary)")
print("=" * 70)
print(f"\nOverall Accuracy: {report['accuracy']*100:.2f}%")
print(f"Macro Average Precision: {report['macro avg']['precision']*100:.2f}%")
print(f"Macro Average Recall:    {report['macro avg']['recall']*100:.2f}%")
print(f"Macro Average F1-Score:  {report['macro avg']['f1-score']*100:.2f}%")
print(f"Weighted Average F1:     {report['weighted avg']['f1-score']*100:.2f}%")

# Print per-class results
print("\n" + "=" * 70)
print("PER-CLASS RESULTS")
print("=" * 70)
print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-" * 66)
for cls in class_names:
    if cls in report:
        r = report[cls]
        print(f"{cls:<20} {r['precision']*100:>8.1f}%   {r['recall']*100:>8.1f}%   "
              f"{r['f1-score']*100:>8.1f}%   {r['support']:>6}")

# Save full report
full_report_text = classification_report(
    test_labels, test_preds,
    target_names=class_names,
    zero_division=0
)
with open('classification_report.txt', 'w') as f:
    f.write(full_report_text)
print("\nFull report saved to 'classification_report.txt'")

In [ ]:
# Cell 30: Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)

# Plot confusion matrix (for all classes - might be large)
fig, ax = plt.subplots(figsize=(25, 25))
sns.heatmap(
    cm, annot=False, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    ax=ax
)
ax.set_xlabel('Predicted', fontsize=14)
ax.set_ylabel('True', fontsize=14)
ax.set_title('Confusion Matrix (All Classes)', fontsize=16)
plt.xticks(rotation=90, fontsize=5)
plt.yticks(rotation=0, fontsize=5)
plt.tight_layout()
plt.savefig('confusion_matrix_full.png', dpi=200, bbox_inches='tight')
plt.show()

# Plot top-20 most confused pairs
print("\nTop 20 Most Confused Class Pairs:")
print("-" * 50)
confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i][j] > 0:
            confused_pairs.append((class_names[i], class_names[j], cm[i][j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)
for true_cls, pred_cls, count in confused_pairs[:20]:
    print(f"  {true_cls:<15} → {pred_cls:<15} : {count} times")

In [ ]:
# Cell 31: Per-class accuracy analysis
per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1) * 100

# Best performing classes
sorted_idx = np.argsort(per_class_acc)[::-1]

print("\n🏆 TOP 10 BEST PERFORMING CLASSES:")
for i in range(min(10, len(sorted_idx))):
    idx = sorted_idx[i]
    print(f"  {class_names[idx]:<20}: {per_class_acc[idx]:.1f}%")

print("\n⚠ TOP 10 WORST PERFORMING CLASSES:")
for i in range(min(10, len(sorted_idx))):
    idx = sorted_idx[-(i+1)]
    print(f"  {class_names[idx]:<20}: {per_class_acc[idx]:.1f}%")

# Plot per-class accuracy
fig, ax = plt.subplots(figsize=(20, 6))
colors = ['green' if acc > 80 else 'orange' if acc > 50 else 'red' for acc in per_class_acc[sorted_idx]]
ax.bar(range(len(per_class_acc)), per_class_acc[sorted_idx], color=colors, alpha=0.7)
ax.axhline(y=test_acc, color='blue', linestyle='--', label=f'Overall: {test_acc:.1f}%')
ax.set_xlabel('Class (sorted by accuracy)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Test Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 32: Single image prediction function
def predict_image(model, image_path, transform, label_encoder, device, top_k=5):
    """Predict the class of a single image."""
    model.eval()

    # Load and transform image
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = F.softmax(output, dim=1)
        top_probs, top_indices = torch.topk(probabilities, top_k)

    # Convert to readable format
    results = []
    for prob, idx in zip(top_probs[0], top_indices[0]):
        class_name = label_encoder.classes_[idx.item()]
        results.append((class_name, prob.item() * 100))

    return image, results


# Visualize predictions on test images
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
sample_indices = random.sample(range(len(X_test)), 12)

for idx, ax in zip(sample_indices, axes.flatten()):
    image, predictions = predict_image(
        model, X_test[idx], val_transform, label_encoder, device, top_k=3
    )
    true_label = label_encoder.classes_[y_test[idx]]

    ax.imshow(image)
    pred_text = f"True: {true_label}\n"
    for cls, prob in predictions:
        marker = "✓" if cls == true_label else "✗"
        pred_text += f"{marker} {cls}: {prob:.1f}%\n"

    color = 'green' if predictions[0][0] == true_label else 'red'
    ax.set_title(pred_text, fontsize=8, color=color, loc='left')
    ax.axis('off')

plt.suptitle('Sample Predictions on Test Set', fontsize=16)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 33 (IMPROVED): Save model in a more compatible way
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'img_size': IMG_SIZE,
        'num_classes': int(num_classes),  # Convert to native Python int
        'embed_dim': 192,
        'depth': (3, 4, 8),
        'num_heads': (3, 6, 12),
    },
    'label_encoder_classes': list(label_encoder.classes_),  # Convert to list
    'best_val_acc': float(best_val_acc),    # Convert numpy → Python float
    'test_acc': float(test_acc),
    # Don't save training_history with numpy values
}, 'visformer_brahmi_complete.pth')

print("✅ Model saved with compatible types!")

In [ ]:
# Cell 34: Download saved files (optional)
from google.colab import files

# Download key files
files_to_download = [
    'best_model.pth',
    'visformer_brahmi_complete.pth',
    'training_history.json',
    'label_mapping.json',
    'classification_report.txt',
    'training_curves.png',
    'confusion_matrix_full.png',
    'sample_predictions.png',
]

for f in files_to_download:
    if os.path.exists(f):
        try:
            files.download(f)
            print(f"Downloaded: {f}")
        except:
            print(f"Available for download: {f}")

**Viewing the weights**

In [ ]:
# Cell: Visualize actual weights in your trained model
import matplotlib.pyplot as plt
import numpy as np

# Load the trained model
checkpoint = torch.load('best_model.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# ===== 1. Look at raw weight numbers =====
print("=" * 60)
print("ACTUAL WEIGHT VALUES IN YOUR MODEL")
print("=" * 60)

for name, param in model.named_parameters():
    print(f"\n{name}:")
    print(f"  Shape: {list(param.shape)}")
    print(f"  Total weights: {param.numel():,}")
    print(f"  Min value:  {param.min().item():.6f}")
    print(f"  Max value:  {param.max().item():.6f}")
    print(f"  Mean value: {param.mean().item():.6f}")
    print(f"  Sample weights: {param.flatten()[:5].tolist()}")

    # Only print first 10 layers to avoid too much output
    if list(model.named_parameters()).index((name, param)) > 10:
        print("\n... (more layers) ...")
        break

# ===== 2. Visualize first layer filters =====
print("\n" + "=" * 60)
print("VISUALIZING FIRST LAYER FILTERS (what the model looks for)")
print("=" * 60)

# Get first conv layer weights
first_conv = None
for name, param in model.named_parameters():
    if 'conv' in name.lower() and 'weight' in name.lower():
        if param.dim() == 4:  # Conv2d weight
            first_conv = param.data.cpu()
            print(f"First conv layer: {name}, shape: {list(first_conv.shape)}")
            break

if first_conv is not None:
    # Visualize first 32 filters
    n_filters = min(32, first_conv.shape[0])
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    fig.suptitle('First Layer Filters (What the Model Looks For)', fontsize=14)

    for i, ax in enumerate(axes.flatten()):
        if i < n_filters:
            # Take the filter and normalize for display
            filt = first_conv[i]
            if filt.shape[0] == 3:  # RGB filter
                filt = filt.permute(1, 2, 0)  # CHW -> HWC
                filt = (filt - filt.min()) / (filt.max() - filt.min())
                ax.imshow(filt.numpy())
            else:
                ax.imshow(filt[0].numpy(), cmap='RdBu')
            ax.set_title(f'Filter {i}', fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('learned_filters.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Each small image is a FILTER (set of weights)")
    print("The model learned these patterns automatically!")

# ===== 3. Weight distribution =====
all_weights = []
for name, param in model.named_parameters():
    if 'weight' in name:
        all_weights.extend(param.data.cpu().flatten().numpy())

all_weights = np.array(all_weights)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(all_weights, bins=100, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Weight Value')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Distribution of All {len(all_weights):,} Weights')
axes[0].axvline(x=0, color='red', linestyle='--', label='Zero')
axes[0].legend()

# Box plot per layer
layer_weights = {}
for name, param in model.named_parameters():
    if 'weight' in name and param.dim() >= 2:
        # Shorten name for display
        short_name = name.split('.')[-2] + '.' + name.split('.')[-1]
        layer_weights[short_name] = param.data.cpu().flatten().numpy()
        if len(layer_weights) >= 10:
            break

axes[1].boxplot(layer_weights.values(), labels=layer_weights.keys())
axes[1].set_ylabel('Weight Value')
axes[1].set_title('Weight Distribution Per Layer')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('weight_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nWeight Statistics:")
print(f"  Total weights: {len(all_weights):,}")
print(f"  Mean: {all_weights.mean():.6f}")
print(f"  Std:  {all_weights.std():.6f}")
print(f"  Most weights are close to 0 (small adjustments)")
print(f"  This is healthy! Very large weights = overfitting")